# Phòng thử ảnh: nén rồi nhận diện trang phục
**Sau bài 10 · 4–5 giờ, làm cá nhân hoặc cặp.** Đây là thời gian dự án bổ sung, không cộng vào 150 phút của bài học.

**Sản phẩm:** So sánh PCA + logistic và PCA + MLP, tái dựng ảnh, xuất mô hình và hàm dự đoán ảnh 28×28.

Đọc → dự đoán → chạy mã mẫu → tự điền TODO → kiểm thử → giải thích. Các TODO cố ý dừng bằng `NotImplementedError`; đây không phải lỗi môi trường. Không thay kiểm thử bằng giá trị viết cứng. Giữ seed, Restart Kernel và chạy lại trước khi nộp.

Nguồn, điều kiện sử dụng và giới hạn dữ liệu: [bảng nguồn](../../course/DATA_SOURCES.md). Không tải mô hình pickle/joblib không rõ nguồn.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'mlcourse').is_dir()), None)
assert ROOT is not None, 'Mở notebook trong bản tải đầy đủ của repository.'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from threadpoolctl import threadpool_limits
_threads = threadpool_limits(limits=2)
SEED = 42

OUT = ROOT/'outputs/projects/05_fashion_lab'
OUT.mkdir(parents=True, exist_ok=True)


## 1. Từ 784 số đến biểu diễn gọn
Fashion-MNIST gồm ảnh xám 28×28 của 10 loại trang phục. Đây không phải ảnh chụp điện thoại ngoài đời. PCA tối đa hóa phương sai giữ lại, không trực tiếp tối đa hóa độ chính xác phân loại.
Thiết kế cố định trước: 3.000 train, 1.000 validation phân tầng từ tập train gốc; test chính thức chỉ mở ở cuối. Hai k:16,32; hai mô hình: logistic và MLP. Chọn theo accuracy validation; hòa chọn cấu hình xuất hiện trước.


In [ ]:
from mlcourse.fashion import load_fashion, NAMES
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib
data=load_fashion(ROOT/'course/bai09/data/raw',SEED,include_test=False)
it,_=train_test_split(np.arange(len(data['train'])),train_size=3000,stratify=data['y_train'],random_state=SEED)
iv,_=train_test_split(np.arange(len(data['validation'])),train_size=1000,stratify=data['y_validation'],random_state=SEED)
Xt,yt=data['train'][it],data['y_train'][it]
Xv,yv=data['validation'][iv],data['y_validation'][iv]
print('Shapes:',Xt.shape,Xv.shape)
print('Kiểm tra trùng pixel trên tập học liệu:',data['duplicate_audit'])
pd.DataFrame({'source_train_row':data['train_ids'][it]}).to_csv(OUT/'train_ids.csv',index=False)
pd.DataFrame({'source_train_row':data['validation_ids'][iv]}).to_csv(OUT/'validation_ids.csv',index=False)


## 2. TODO: đo mất mát tái dựng
MSE trung bình trên **tất cả ảnh và pixel**. Hai mảng cùng shape (n,784), thang [0,1]. Tự tính rồi đối chiếu bằng ví dụ nhỏ.


In [ ]:
def reconstruction_mse(x, reconstructed):
    x,reconstructed=np.asarray(x,float),np.asarray(reconstructed,float)
    if x.shape!=reconstructed.shape: raise ValueError('Sai kích thước')
    # TODO: trung bình bình phương sai khác.
    raise NotImplementedError('Hoàn thiện MSE')


In [ ]:
assert np.isclose(reconstruction_mse([[0,1]],[[0,.5]]),.125)
assert reconstruction_mse([[1,2]],[[1,2]])==0


## 3. TODO: PCA chỉ học từ train
Trả (pca, Ztrain, Zvalidation). Dùng PCA(n_components=k, svd_solver="randomized", random_state=SEED), fit Xt, transform Xt/Xv. PCA của sklearn tự trừ trung bình train; không fit lại trên validation.


In [ ]:
def compress(k):
    # TODO: fit Xt; transform Xt và Xv.
    raise NotImplementedError('Hoàn thiện PCA')


In [ ]:
pca,Zt,Zv=compress(16)
assert Zt.shape==(len(Xt),16) and Zv.shape==(len(Xv),16)
np.testing.assert_allclose(pca.mean_,Xt.mean(axis=0),atol=1e-10)
reconstructed=pca.inverse_transform(Zv)
print('Validation reconstruction MSE:',reconstruction_mse(Xv,reconstructed))
fig,axes=plt.subplots(2,6,figsize=(10,4))
for i in range(6):
    axes[0,i].imshow(Xv[i].reshape(28,28),cmap='gray',vmin=0,vmax=1)
    axes[1,i].imshow(reconstructed[i].reshape(28,28),cmap='gray',vmin=0,vmax=1)
    axes[0,i].axis('off');axes[1,i].axis('off')
plt.savefig(OUT/'reconstruction.png');plt.show()


## 4. Mã mẫu: so sánh nhỏ, ngân sách cố định
MLP có một lớp ẩn 64 neuron, 40 epoch tối đa. Cảnh báo chưa hội tụ cần ghi trong báo cáo; không được ẩn cảnh báo rồi tuyên bố đã hội tụ. Không thay ngân sách theo kết quả test. Hai mô hình dùng cùng PCA/train/validation.


In [ ]:
import warnings
rows=[]; candidates=[]
for k in [16,32]:
    pca,Zt,Zv=compress(k)
    for name,clf in [('logistic',LogisticRegression(max_iter=300,random_state=SEED)),('mlp',MLPClassifier(hidden_layer_sizes=(64,),max_iter=40,random_state=SEED,early_stopping=False))]:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always');clf.fit(Zt,yt)
        accuracy=float(accuracy_score(yv,clf.predict(Zv)))
        rows.append({'k':k,'model':name,'validation_accuracy':accuracy,'warnings':'; '.join(str(w.message) for w in caught)})
        candidates.append((pca,clf))
comparison=pd.DataFrame(rows);display(comparison)
best=int(np.argmax(comparison.validation_accuracy.to_numpy()))
best_pca,best_model=candidates[best]
comparison.to_csv(OUT/'validation_comparison.csv',index=False)
print('Cấu hình đã khóa:',comparison.iloc[best].to_dict())


## 5. Mở test và xuất sản phẩm
Đây là test công khai đã được dùng trong học liệu; kết quả là thực hành tái lập, không phải benchmark trên dữ liệu niêm phong mới. Không đổi cấu hình sau khi xem test.


In [ ]:
full=load_fashion(ROOT/'course/bai09/data/raw',SEED,include_test=True)
Xe,ye=full['test'],full['y_test']
pred=best_model.predict(best_pca.transform(Xe))
print(classification_report(ye,pred,target_names=NAMES,zero_division=0))
metrics={'test_accuracy':float(accuracy_score(ye,pred)),'selected':rows[best],'train_count':len(yt),'validation_count':len(yv),'test_count':len(ye),'seed':SEED,'duplicate_audit':full['duplicate_audit']}
(OUT/'metrics.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
pd.DataFrame({'test_row':np.arange(len(ye)),'actual':ye,'predicted':pred}).to_csv(OUT/'predictions.csv',index=False)
joblib.dump({'pca':best_pca,'model':best_model,'classes':NAMES},OUT/'fashion_model.joblib')


## 6. Hàm nhận ảnh mới và giới hạn
Hàm chỉ chấp nhận ảnh xám 28×28 đã chuẩn hóa [0,1]. Không tự reshape ảnh bất kỳ và cho rằng nó giống dữ liệu train.


In [ ]:
bundle=joblib.load(OUT/'fashion_model.joblib')
def predict_image(image):
    image=np.asarray(image,float)
    if image.shape!=(28,28) or not np.isfinite(image).all() or image.min()<0 or image.max()>1:
        raise ValueError('Cần ảnh 28x28 hữu hạn, trong [0,1]')
    label=int(bundle['model'].predict(bundle['pca'].transform(image.reshape(1,-1)))[0])
    return bundle['classes'][label]
print(predict_image(Xe[0].reshape(28,28)))
assert predict_image(Xe[0].reshape(28,28))==NAMES[pred[0]]


## Hoàn thiện sản phẩm và tự kiểm
MLP không được giả định luôn tốt hơn logistic. Mở rộng: so sánh ảnh tái dựng theo k và phân tích các cặp lớp nhầm; một mạng học trên toàn bộ pixel là thí nghiệm riêng, cần tiêu chí và ngân sách rõ ràng.

Nộp notebook đã chạy, thư mục đầu ra của dự án và `REPORT.md` tự viết: vấn đề → dữ liệu → phương pháp → kết quả → giới hạn. Dẫn nguồn, ghi seed/cách chia và mô tả ít nhất một trường hợp thất bại. Không chỉ chụp hình điểm số.

**Câu hỏi phản tư:** Điều gì trong kết quả là bằng chứng, điều gì mới là giả thuyết? Nếu dữ liệu mới khác dữ liệu học, bạn cần kiểm tra gì?

**Nhiệm vụ mở:** thay đổi một lựa chọn có lý do, xác định tiêu chí trước khi chạy và so sánh trên validation (hoặc phân tích độ ổn định với bài phân cụm). Không liên tục xem test để chọn cách làm.
